# Ant Colony Optimization para TSP

Implementación compacta para observar construcción probabilística, evaporación y depósito de feromona.

## Dependencias

En un entorno nuevo: `%pip install numpy matplotlib`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng_instancia = np.random.default_rng(2026)
ciudades = rng_instancia.uniform(0, 100, size=(24, 2))
distancias = np.linalg.norm(ciudades[:, None, :] - ciudades[None, :, :], axis=2)
np.fill_diagonal(distancias, np.inf)

In [ ]:
def longitud(tour):
    return sum(distancias[tour[i], tour[(i + 1) % len(tour)]] for i in range(len(tour)))

def construir_tour(rng, feromona, alpha, beta):
    n = len(feromona)
    tour = [int(rng.integers(n))]
    no_visitadas = set(range(n)) - set(tour)
    while no_visitadas:
        actual = tour[-1]
        opciones = np.array(sorted(no_visitadas))
        pesos = feromona[actual, opciones] ** alpha * (1 / distancias[actual, opciones]) ** beta
        probabilidades = pesos / pesos.sum()
        siguiente = int(rng.choice(opciones, p=probabilidades))
        tour.append(siguiente)
        no_visitadas.remove(siguiente)
    return tour

In [ ]:
def ejecutar_aco(semilla, iteraciones=180, n_hormigas=30, alpha=1.0, beta=3.0, rho=0.25, q=100.0):
    rng = np.random.default_rng(semilla)
    n = len(ciudades)
    feromona = np.ones((n, n), dtype=float)
    mejor_tour, mejor_largo = None, np.inf
    historial = []
    for _ in range(iteraciones):
        tours = [construir_tour(rng, feromona, alpha, beta) for _ in range(n_hormigas)]
        largos = np.array([longitud(tour) for tour in tours])
        indice = int(np.argmin(largos))
        if largos[indice] < mejor_largo:
            mejor_tour, mejor_largo = tours[indice].copy(), float(largos[indice])
        feromona *= 1 - rho
        for tour, largo in zip(tours, largos):
            deposito = q / largo
            for i, origen in enumerate(tour):
                destino = tour[(i + 1) % n]
                feromona[origen, destino] += deposito
                feromona[destino, origen] += deposito
        historial.append(mejor_largo)
    return mejor_tour, mejor_largo, np.asarray(historial)

In [ ]:
tour, largo, historial = ejecutar_aco(semilla=7)
print(f'Mejor longitud: {largo:.2f}')

ruta = np.array(tour + [tour[0]])
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(historial)
axes[0].set(title='Convergencia', xlabel='Iteración', ylabel='Mejor longitud')
axes[1].plot(ciudades[ruta, 0], ciudades[ruta, 1], '-o')
axes[1].set(title=f'Mejor tour: {largo:.2f}', aspect='equal')
for ax in axes: ax.grid(alpha=0.3)
plt.show()

## Variabilidad y línea base

In [ ]:
resultados = [ejecutar_aco(s, iteraciones=100)[1] for s in range(10)]
rng = np.random.default_rng(0)
aleatorios = [longitud(list(rng.permutation(len(ciudades)))) for _ in range(1000)]
print('ACO mediana:', np.median(resultados))
print('Aleatorio mediana:', np.median(aleatorios))
print('ACO rango:', min(resultados), max(resultados))

## Trabajo propuesto

Compare valores de $\alpha$, $\beta$ y $\rho$ con el mismo número de tours evaluados. Agregue una heurística nearest-neighbor como línea base más exigente.